# Fiscozen Finance Analysis

Questo notebook analizza la performance fiscale di Fiscozen degli abbonamenti venduti tra gennaio e febbraio 2026

## 1. Import Libraries

In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.2f}'.format
import matplotlib.pyplot as plt
import numpy as np

## 2. Load Dataset

In [2]:
df = pd.read_csv("subscriptions.csv")
df.shape
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   subscription_id  5000 non-null   int64  
 1   start_date       5000 non-null   object 
 2   end_date         5000 non-null   object 
 3   payment_date     5000 non-null   object 
 4   invoice_amount   5000 non-null   float64
 5   booking_amount   5000 non-null   float64
 6   refund_date      82 non-null     object 
 7   refund_amount    5000 non-null   float64
 8   kind             5000 non-null   object 
dtypes: float64(3), int64(1), object(5)
memory usage: 351.7+ KB


,subscription_id,start_date,end_date,payment_date,invoice_amount,booking_amount,refund_date,refund_amount,kind
0,202262,2026-01-19,2027-01-19,2026-01-19,374.02,374.02,NaN,0.00,Upfront
1,206816,2026-01-29,2027-01-30,2026-01-29,374.02,374.02,NaN,0.00,Upfront
2,202264,2026-01-19,2027-01-19,2026-01-19,455.98,455.98,NaN,0.00,Upfront
3,212828,2026-02-15,2027-02-15,2026-02-15,292.56,292.56,NaN,0.00,Upfront
4,200663,2026-01-16,2027-01-16,2026-01-16,374.02,374.02,NaN,0.00,Upfront


## 3. Data Cleaning

### 1. Conversione string a date

In [3]:
date_cols = ["start_date", "end_date", "payment_date", "refund_date"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  5000 non-null   int64         
 1   start_date       5000 non-null   datetime64[ns]
 2   end_date         5000 non-null   datetime64[ns]
 3   payment_date     5000 non-null   datetime64[ns]
 4   invoice_amount   5000 non-null   float64       
 5   booking_amount   5000 non-null   float64       
 6   refund_date      82 non-null     datetime64[ns]
 7   refund_amount    5000 non-null   float64       
 8   kind             5000 non-null   object        
dtypes: datetime64[ns](4), float64(3), int64(1), object(1)
memory usage: 351.7+ KB


### 2. Controllo in possibili errori tipologia di pagamento

In [4]:
df["kind"].value_counts()

kind
Upfront    4998
Monthly       2
Name: count, dtype: int64

### 3. Controllo valori unici in subscription_ip

In [5]:
df.duplicated(subset="subscription_id").sum()

np.int64(0)

### 4. Controllo date comprese tra gennaio e febbraio 2026 e differenza date 365

In [6]:
len(df[df["start_date"] < "2026-01-01"])

9

9 subscriptions were excluded because their start date precedes the analysis period.

In [7]:
len(df[df["start_date"] > "2026-02-28"])

0

In [8]:
df["start_date"].dt.to_period("M").value_counts().sort_index()
df = df[df["start_date"] >= "2026-01-01"]
df["start_date"].min()

Timestamp('2026-01-01 00:00:00')

In [9]:
df["service_days"] = (df["end_date"] - df["start_date"]).dt.days
(df["service_days"] != 365).sum()

np.int64(28)

Ci sono 28 righe con una differenza di date che non è di 365 gg tra la data di inizio e fine servizio. Le correggiamo

In [10]:
df["end_date"] = df["start_date"] + pd.DateOffset(years=1)
df["service_days"] = (df["end_date"] - df["start_date"]).dt.days
(df["service_days"] != 365).sum()

np.int64(0)

### 5. Controllo che solo in caso di pagamenti mensili booking_amount > invoice_amount

In [11]:
df.groupby("kind")[["booking_amount","invoice_amount"]].mean()

,booking_amount,invoice_amount
kind,,
Monthly,421.92,52.74
Upfront,361.81,361.81


Per gli abbonamenti con pagamento Upfront, l’intero valore dell’abbonamento viene fatturato immediatamente, quindi booking_amount e invoice_amount coincidono.

Per gli abbonamenti con pagamento Monthly, invece, il cliente paga in 12 rate mensili: di conseguenza booking_amount rappresenta il valore totale della vendita, mentre invoice_amount rappresenta solo la quota già fatturata fino ad oggi.

## 4. Booking Analysis

Consideriamo come venduto il valore totale dell’abbonamento venduto e non quello riscosso

In [12]:
df["payment_month"] = df["payment_date"].dt.to_period("M")

bookings_analysis = df.groupby("payment_month").agg(
    Bookings=("booking_amount", "sum"),
    Subscriptions=("subscription_id", "count")
)

bookings_analysis["Average Booking Value per Subscription"] = (
    bookings_analysis["Bookings"] / bookings_analysis["Subscriptions"]
)

bookings_analysis

,Bookings,Subscriptions,Average Booking Value per Subscription
payment_month,,,
2026-01,"1,230,465.69",3389,363.08
2026-02,"575,462.34",1602,359.21


Il venduto è significativamente più alto a gennaio (€1,23M) rispetto a febbraio (€0,58M). Questo potrebbe essere legato alla natura del servizio offerto: molti professionisti rinnovano, aprono o cambiano gestione fiscale all’inizio dell’anno, generando una maggiore domanda nel mese di gennaio.

Il numero di abbonamenti segue lo stesso andamento del venduto: 3.389 sottoscrizioni a gennaio contro 1.602 a febbraio. Questo conferma che la maggiore performance di gennaio è guidata principalmente da un volume più alto di nuovi clienti, probabilmente legato all’inizio dell’anno fiscale.

L'importo medio del venduto per abbonamento è rimasto costante nei due periodi (363€/sbs vs 359€/sbs). Si può quindi concludere che la crescita del venduto è stata interamente trainata dalla formulazione di nuovi abbonamenti. 

## 5. Refound Management

In [13]:
df["net_booking"] = df["booking_amount"] - df["refund_amount"]
net_bookings_by_month = df.groupby("payment_month")["net_booking"].sum()

net_bookings_by_month

payment_month
2026-01   1,213,809.85
2026-02     563,717.70
Freq: M, Name: net_booking, dtype: float64

### 1. Number and Value of refound

In [14]:
refund_count = (df["refund_amount"] > 0).sum()
refund_count

np.int64(80)

In [15]:
total_refunds = df["refund_amount"].sum()
total_refunds

np.float64(28400.48)

In [16]:
refund_rate = refund_count / len(df)
refund_rate

np.float64(0.016028851933480266)

Nel complesso, 1,60% di refund rate indica una buona retention iniziale dei clienti durante i primi 30 giorni di servizio.

### 2. Distribution per month

In [17]:
summary_by_month = df.groupby("payment_month").agg(
    total_bookings=("booking_amount", "sum"),
    total_refunds=("refund_amount", "sum"),
    refund_count=("refund_amount", lambda x: (x > 0).sum()),
    transactions=("refund_amount", "count")
)

summary_by_month["net_bookings"] = summary_by_month["total_bookings"] - summary_by_month["total_refunds"]
summary_by_month["refund_rate"] = summary_by_month["refund_count"] / summary_by_month["transactions"]

summary_by_month

,total_bookings,total_refunds,refund_count,transactions,net_bookings,refund_rate
payment_month,,,,,,
2026-01,"1,230,465.69","16,655.84",50,3389,"1,213,809.85",0.01
2026-02,"575,462.34","11,744.64",30,1602,"563,717.70",0.02


Il tasso di rimborso rimane basso e relativamente stabile, c'è un leggero incremento a febbraio, tra i due mesi analizzati: 1,48% a gennaio e 1,87% a febbraio.

## 6. Cashed In

Cash netta = incassi − rimborsi. 
Ovviamente come incassi consideriamo gli invoice e non bookings

In [18]:
df["net_invoice"] = df["invoice_amount"] - df["refund_amount"]

In [19]:
cash_analysis = df.groupby("payment_month").agg(
    Bookings=("booking_amount", "sum"),
    Invoice_amount=("invoice_amount", "sum"),
    Refund=("refund_amount", "sum"),
    Cash_Collected=("net_invoice", "sum"),
)
cash_analysis["Cash collection rate"] = cash_analysis["Cash_Collected"] / cash_analysis["Bookings"]

cash_analysis

,Bookings,Invoice_amount,Refund,Cash_Collected,Cash collection rate
payment_month,,,,,
2026-01,"1,230,465.69","1,229,727.33","16,655.84","1,213,071.49",0.99
2026-02,"575,462.34","575,462.34","11,744.64","563,717.70",0.98


La cassa incassata è molto vicina al venduto in entrambi i mesi analizzati. Il cash collection rate è pari a circa 98,6% a gennaio e 98% a febbraio, indicando che la quasi totalità delle vendite viene fatturata e incassata nel mese della sottoscrizione. Buona conversione del venduto in cassa immediata. Il modello di pagamento Monthly ha un impatto molto limitato
sulla cassa perchè la maggior parte degli utenti sceglie il pagamento upfront.

## 7. Deferred Revenue

In [20]:
df["daily_revenue"] = df["net_invoice"] / 365

months = pd.period_range("2026-01", "2026-02", freq="M")

results = []

for month in months:

    month_start = month.start_time
    month_end = month.end_time

    start = df["start_date"].clip(lower=month_start)
    end = df["end_date"].clip(upper=month_end)

    days = (end - start).dt.days.clip(lower=0)

    revenue = (days * df["daily_revenue"]).sum()

    bookings_month = df.loc[df["payment_month"] == month, "net_booking"].sum()
    invoice_month = df.loc[df["payment_month"] == month, "net_invoice"].sum()

    results.append({
        "Mese": month.strftime("%B"),
        "Venduto netto": bookings_month,
        "Fatturato netto": invoice_month,
        "Ricavo riconosciuto": revenue
    })

revenue_by_month = pd.DataFrame(results)

revenue_by_month["Ricavi differiti"]=revenue_by_month["Fatturato netto"] - revenue_by_month["Ricavo riconosciuto"]
revenue_by_month["ratio_ricavo_riconosciuto"]= revenue_by_month["Ricavo riconosciuto"]/revenue_by_month["Fatturato netto"]

revenue_by_month

,Mese,Venduto netto,Fatturato netto,Ricavo riconosciuto,Ricavi differiti,ratio_ricavo_riconosciuto
0,January,"1,213,809.85","1,213,071.49","54,942.43","1,158,129.06",0.05
1,February,"563,717.70","563,717.70","111,018.98","452,698.72",0.20


La maggior parte del venduto è ancora deferred revenue. La quota di subscription già incassata ma non ancora erogata viene registrata come deferred revenue (risconti passivi) nello Stato Patrimoniale. Questa voce rappresenta una passività, che verrà progressivamente ridotta man mano che il servizio viene erogato e il ricavo viene riconosciuto nel conto economico.